# Personal Information
Name: **Alexandra Holíková**

StudentID: **14236788**

Email: [**alexandra.holikova@student.uva.nl**](alexandra.holikova@student.uva.nl)

Github link: https://github.com/alexandraholik/MSc_Thesis.git

# English set construction

This notebook constructs the English-language annotation set used in the thesis. The goal is to identify news articles in which a quoted span in the headline can be compared to quoted speech in the article body.

The notebook follows four main steps:
1. Stream and filter CC-News articles.
2. Extract quoted spans from headlines and article bodies.
3. Remove cases where the headline quote is near-verbatim repetition of a body quote.
4. Sample a final subset for manual annotation.

In [2]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from datasets import load_dataset
from urllib.parse import urlparse  

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 120)

# Paths
DATA_DIR = "data_new"
os.makedirs(DATA_DIR, exist_ok=True)
print(f"Data directory: {os.path.abspath(DATA_DIR)}")

Data directory: /Users/alexandraholikova/PycharmProjects/master_thesis_holikova/data_new


## Data source and collection

This section loads articles from the `stanford-oval/ccnews` dataset in streaming mode. To keep the dataset manageable while preserving scale, only the first portion of the stream is scanned.

Only articles are retained that:
- are labeled as English,
- have a language confidence score of at least 0.9,
- contain a non-empty headline and body text,
- contain at least 100 characters of body text.

For each retained article, the notebook stores metadata needed for later filtering and annotation, including publication date, domain, title, body text, URL, publisher, and sitename.

In [ ]:
SCAN_MAX = 800_000
REPORT_N = 10_000

print("Opening stanford-oval/ccnews stream …")
stream = load_dataset(
    "stanford-oval/ccnews",
    name="default",
    split="train",
    streaming=True,
)

rows, seen = [], 0
print(f"Collecting up to {SCAN_MAX:,} articles …\n")

for article in stream:
    seen += 1
    if seen % REPORT_N == 0:
        print(f"  … {seen:,} scanned | {len(rows):,} collected")

    if article.get("language") != "en":
        continue
    if (article.get("language_score") or 0) < 0.9:
        continue
        
    if not article.get("title") or not article.get("plain_text"):
        continue
    if len(article["plain_text"].strip()) < 100:
        continue

    url = article.get("requested_url") or article.get("responded_url") or ""
    domain_norm = urlparse(url).netloc.lower()
    domain_norm = re.sub(r"^www\.", "", domain_norm)

    rows.append({
        "date":        article.get("published_date"),
        "domain":      urlparse(url).netloc,       #raw, for reference
        "domain_norm": domain_norm,                #normalised, for filtering
        "title":       article["title"].strip(),
        "text":        article["plain_text"].strip(),
        "url":         url,
        "publisher":   article.get("publisher"),
        "sitename":    article.get("sitename"),
    })

    if seen >= SCAN_MAX:
        break

df = pd.DataFrame(rows)

print(f"\n✓ Scanned : {seen:,} articles")
print(f"✓ Collected (non-empty, English): {len(df):,}")
print(f"  Unique domains : {df['domain_norm'].nunique():,}")
print(f"  Date range     : {df['date'].min()} → {df['date'].max()}")
df.head(2)

Opening stanford-oval/ccnews stream …


Resolving data files:   0%|          | 0/479 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/479 [00:00<?, ?it/s]

## Intermediate dataset storage

After the initial filtering stage, the retained articles are saved to disk as a CSV file. This makes the later stages reproducible and avoids rerunning the full data stream each time the notebook is opened.

In [ ]:
csv_path = os.path.join(DATA_DIR, "cc_news_full.csv")
df.to_csv(csv_path, index=False)
size_mb = os.path.getsize(csv_path) / 1e6
print(f"Saved to {csv_path}  ({size_mb:.0f} MB)")
print(f"Shape: {df.shape}")

In [3]:
df = pd.read_csv("data_new/cc_news_full.csv")

## Quote extraction

This section identifies quoted spans in both headlines and article bodies.

Two different extraction strategies are used:
- **Headline quotes** are extracted if they appear inside quotation marks and contain at least four words.
- **Body quotes** are extracted only if they contain at least four words and occur near an attribution cue such as `said`, `told`, or `argued`.

The stricter rule for body quotes is intended to favor likely instances of direct speech rather than arbitrary quoted fragments.

In [4]:

# Double-quoted spans: "..." or "..."
DQUOTE_RE = re.compile(
    "[\"\u201c]([^\"\u201c\u201d]{20,300})[\"\u201d]"
)
# Single-quoted spans: '...' or '...'
# Boundary-aware to avoid matching apostrophes inside words
SQUOTE_RE = re.compile(
    "(?:^|[\\s\\-\u2014\u2013:])['\u2018]([^'\u2018\u2019]{20,300})['\u2019](?=[\\s.,;:!?\\-\u2014\u2013]|$)"
)
# Attribution verbs for filtering body quotes to likely direct speech
ATTRIB_RE = re.compile(
    r'\b(said|says|told|telling|asked|added|argued|claimed|explained|'
    r'noted|warned|insisted|suggested|recalled|admitted|declared|'
    r'announced|commented|replied|responded|continued|wrote|tweeted|'
    r'posted|stated|described|called|according)\b',
    re.IGNORECASE
)

MIN_QUOTE_WORDS = 4


def extract_headline_quotes(text):
    """Extract quoted spans from headline. 4-word minimum, no attribution check."""
    if not isinstance(text, str):
        return []
    raw = DQUOTE_RE.findall(text) + SQUOTE_RE.findall(text)
    return [q for q in raw if len(q.split()) >= MIN_QUOTE_WORDS]


def extract_speech_quotes(text):
    """Extract body quoted spans likely to be direct speech.
    Requires 4+ words AND an attribution verb within 80 chars."""
    if not isinstance(text, str):
        return []
    quotes = []
    # find spans with their positions so we can check the surrounding window
    for pattern in [DQUOTE_RE, SQUOTE_RE]:
        for m in pattern.finditer(text):
            q = m.group(1)
            if len(q.split()) < MIN_QUOTE_WORDS:
                continue
            start = m.start()
            window_start = max(0, start - 80)
            window_end = min(len(text), m.end() + 80)
            window = text[window_start:window_end]
            if ATTRIB_RE.search(window):
                quotes.append(q)
    return quotes


df["title_quotes"] = df["title"].apply(extract_headline_quotes)
df["body_quotes"] = df["text"].apply(extract_speech_quotes)
df["n_title_quotes"] = df["title_quotes"].str.len()
df["n_body_quotes"] = df["body_quotes"].str.len()
df["has_title_quote"] = df["n_title_quotes"] > 0
df["has_body_quote"] = df["n_body_quotes"] > 0

print(f"Total articles               : {len(df):,}")
print(f"With headline quote (4+ words): {df['has_title_quote'].sum():,}  "
      f"({df['has_title_quote'].mean() * 100:.1f}%)")
print(f"With body speech quotes       : {df['has_body_quote'].sum():,}  "
      f"({df['has_body_quote'].mean() * 100:.1f}%)")
print(f"Both (eligible)               : "
      f"{(df['has_title_quote'] & df['has_body_quote']).sum():,}")

eligible = df[df["has_title_quote"] & df["has_body_quote"]].copy()
eligible = eligible.reset_index(drop=True)
print(f"Eligible articles: {len(eligible):,}")
print(f"Mean body quotes per article: {eligible['n_body_quotes'].mean():.1f}")


Total articles               : 306,842
With headline quote (4+ words): 5,166  (1.7%)
With body speech quotes       : 177,798  (57.9%)
Both (eligible)               : 3,927
Eligible articles: 3,927
Mean body quotes per article: 4.9


## Eligibility filtering

An article is considered eligible if it contains:
- at least one sufficiently long quoted span in the headline, and
- at least one likely direct-speech quote in the body.

This filtering step narrows the dataset to articles relevant for the headline-to-body quote comparison task.

## Sampling for annotation

From the filtered pool, a random sample of 2,000 articles is drawn using a fixed seed for reproducibility.

The exported file contains:
- article metadata,
- the headline quote,
- all extracted body quotes,
- the full article text,
- empty columns for annotation labels and notes.

This file serves as the input for the manual annotation stage of the thesis.

In [5]:
#sampling
SAMPLE_N = 2000
SEED = 42

sample = eligible.sample(n=min(SAMPLE_N, len(eligible)), random_state=SEED).reset_index(drop=True)
print(f"Sampled: {len(sample)} articles from {len(eligible):,} pool")

sample_out = pd.DataFrame({
    "article_id": range(1, len(sample) + 1),
    "domain": sample["domain"].values,
    "date": sample["date"].values,
    "url": sample["url"].values,
    "title": sample["title"].values,
    "headline_quote": sample["title_quotes"].apply(lambda q: q[0] if q else "").values,
    "body_quotes": sample["body_quotes"].apply(lambda q: " ||| ".join(q)).values,
    "n_body_quotes": sample["n_body_quotes"].values,
    "full_text": sample["text"].values,
    "label": "",
    "notes": "",
})

out_path = os.path.join(DATA_DIR, "english_annotation_sample_v2.csv")
sample_out.to_csv(out_path, index=False, encoding="utf-8-sig")


Sampled: 2000 articles from 3,927 pool
